# BBC News — Business Category Sub-classification
## NLP Pipeline: BGE-M3 Embeddings + BERTopic (UMAP + HDBSCAN + c-TF-IDF)

**Architecture:**
- Embedding Model: BGE-M3 (BAAI/bge-m3).8192 token context window
- Dimensionality Reduction: UMAP. it compresses 1024 dims to 5
- Clustering: HDBSCAN. This finds natural sub-categories automatically
- Topic Representation: c-TF-IDF. It extracts distinctive keywords per cluster

**Final Output:** DataFrame mapping every article to its sub-category

## Environment Setup & Imports

In [2]:
# Core libraries
import os
import re
import sys
import pandas as pd
import numpy as np

# Path setup — loader.py lives in parent directory
sys.path.append('..')
from loader import load_data

# Sentence Transformers — for BGE-M3
from sentence_transformers import SentenceTransformer

# BERTopic pipeline
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# Visualization
import matplotlib.pyplot as plt

print("All imports successful")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

All imports successful
Pandas version: 2.3.3
NumPy version: 1.23.5


## Data Ingestion Layer
### Loading raw business articles from local storage
### Source: BBC News dataset for business category
### No aggressive cleaning at this stage, BGE-M3 needs full sentence structure

In [3]:
# Load all articles and labels
documents, labels = load_data('../data')

# Filter business articles only
business_docs = [documents[i] for i in range(len(documents)) 
                 if labels[i] == 'business']

# Store filenames for final output
business_files = [f"article_{i}.txt" for i in range(len(business_docs))]

print(f"Total articles in dataset: {len(documents)}")
print(f"Business articles: {len(business_docs)}")
print(f"\nSample article preview:")
print(f"{business_docs[0][:300]}")

Total articles in dataset: 2225
Business articles: 510

Sample article preview:
UK economy facing 'major risks'

The UK manufacturing sector will continue to face "serious challenges" over the next two years, the British Chamber of Commerce (BCC) has said.

The group's quarterly survey of companies found exports had picked up in the last three months of 2004 to their best level


## Light Cleaning (Data Normalization Layer)
### BGE-M3 understands full sentence structure, no aggressive cleaning needed
### We only remove noise that would confuse the transformer:
### - Encoding errors
### - Excessive whitespace
### - HTML tags if present
### Punctuation, stopwords, and proper nouns are intentionally preserved

In [4]:
def light_clean(text):
    # Fix encoding errors
    text = text.encode('utf-8', 'ignore').decode('utf-8')
    
    # Remove HTML tags if present
    text = re.sub(r'<.*?>', '', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply light cleaning to all business articles
cleaned_business = [light_clean(doc) for doc in business_docs]

print(f"Cleaning complete — {len(cleaned_business)} articles processed")
print(f"\nBefore cleaning:")
print(f"{business_docs[0][:200]}")
print(f"\nAfter cleaning:")
print(f"{cleaned_business[0][:200]}")

Cleaning complete — 510 articles processed

Before cleaning:
UK economy facing 'major risks'

The UK manufacturing sector will continue to face "serious challenges" over the next two years, the British Chamber of Commerce (BCC) has said.

The group's quarterly 

After cleaning:
UK economy facing 'major risks' The UK manufacturing sector will continue to face "serious challenges" over the next two years, the British Chamber of Commerce (BCC) has said. The group's quarterly su


##  Deduplication
### Removing duplicate articles before embedding generation
### Duplicates hurt clustering,identical vectors pull clusters toward their content
### We preserve filenames alongside documents to maintain our final output mapping

In [5]:
# Remove duplicates while preserving filename mapping
seen = set()
unique_business = []
unique_files = []

for doc, filename in zip(cleaned_business, business_files):
    if doc not in seen:
        seen.add(doc)
        unique_business.append(doc)
        unique_files.append(filename)

print(f"Before deduplication: {len(cleaned_business)} articles")
print(f"After deduplication:  {len(unique_business)} articles")
print(f"Duplicates removed:   {len(cleaned_business) - len(unique_business)}")

Before deduplication: 510 articles
After deduplication:  503 articles
Duplicates removed:   7
